# import

In [ ]:
from utils.recbole_train_test import *
from utils.plot_utils import *
from utils.model_utils import get_trainer

# Experiment LastFM - ET RD LS.t UD SF TO UM.100
executed train, random drift, leave one out train sampled, uniform neg sample training distribution, shuffle false, time ordering, uniform mode


In [ ]:
SAVE_PATH, BASE_FILENAME, SPECS_STR = ('processed_datasets/natural_data/lastfm1b/tracks_inter_merged_coldstart_3M/',
                                        'more_2interQ_df', 
                                        'UII_1959x98333_1.0')

BASE_DATASET_NAME = BASE_FILENAME+'_'+SPECS_STR

# MODEL_VERSIONS = ['_pt1', '_pt2', '_pt3', '_pt4']
freq=6 # month
duration = 2*12//freq # 2 years split in xM buckets
n_parts = duration*2+1
d_keys = ['_pt'+str(i) for i in range(1, n_parts)]
MODEL_VERSIONS = d_keys[:duration]


K = [1, 10, 20]
VALID_METRIC = 'Recall@'+str(K[1])
SEED = 2020
USE_GPU = False
SHOW_PROGRESS = False

# these are the default values
# TRAIN_NEG_SAMPLE_ARGS = {'distribution': 'uniform', 
#                          'sample_num': 1, 
#                          'alpha': 1.0, 
#                          'dynamic': False, 
#                          'candidate_num': 0}



SHUFFLE = False  # shuffle (bool): Whether or not to shuffle the training data before each epoch. Defaults to True.
EVAL_ARGS = {'split': {'LS': 'test_only'}, # leave-one-out sample type ['valid_and_test', 'valid_only', 'test_only']
                    'group_by': 'user',
                    'order': 'TO', # order (str): decides how we sort the data in .inter. random ordering or time ordering
                    'mode': 'uni100'}

FILENAME_VERSION = '_ET_RD_LS.t_UD_SF_TO_UM.100'

## BPR

In [ ]:
model_name = 'BPR'


for part in MODEL_VERSIONS:
    print('\n\n'+part)
    dataset_name=BASE_DATASET_NAME+part
    parameter_dict = {
        'dataset': dataset_name+'.inter',
        'use_gpu':USE_GPU,

        ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
        'seed':SEED,
        'state':'ERROR',
        'data_path': SAVE_PATH,
        # save_dataset (bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
        'checkpoint_dir':SAVE_PATH+dataset_name,
        # 'show_progress': SHOW_PROGRESS,
        'shuffle': SHUFFLE,

        ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
        'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
        # 'user_inter_num_interval':'[1,inf)',
        
        ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
        
        ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
        'eval_args': EVAL_ARGS,
        # 'metrics': ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision'], # default
        'topk':K,
        'valid_metric':VALID_METRIC          
    }

    recbole_train_evaluate_bpr(model_name, dataset_name, parameter_dict, FILENAME_VERSION)


### evaluate on test set - pt1 without CS and on sectioned parts

In [ ]:
model_name = 'BPR'
model_checkpoint_versions = {'_pt1':model_name+'-Mar--2025_',
                             '_pt2':model_name+'-Mar--2025_',
                             '_pt3':model_name+'-Mar--2025_',
                             '_pt4':model_name+'-Mar--2025_'}


for part in MODEL_VERSIONS:
    print('\n\n'+part)

    current_dataset_name = BASE_DATASET_NAME+part

    # Checkpoint - 
    model_checkpoint_ver = model_checkpoint_versions[part]
    model_checkpoint_dir = SAVE_PATH+BASE_DATASET_NAME+part
    model_checkpoint_file = model_checkpoint_dir+'/'+model_checkpoint_ver+'.pth'


    test_data_sections = get_test_data_sections_with_names(model_version=part,
                                            base_dataset_name=BASE_DATASET_NAME,
                                            models_versions=MODEL_VERSIONS,
                                            pt1NoCS='NoCS')
    print(test_data_sections)
    

    parameter_dict = {
        'dataset': current_dataset_name+'.inter',
        'use_gpu':USE_GPU,

        ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
        'seed':SEED,
        'state':'ERROR',
        'data_path': SAVE_PATH,
        # save_dataset (bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
        'checkpoint_dir':model_checkpoint_dir,
        # 'show_progress': SHOW_PROGRESS,
        'shuffle': SHUFFLE,

        ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
        'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
        # 'user_inter_num_interval':'[1,inf)',
        
        ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
        
        ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
        'eval_args': EVAL_ARGS,
        # 'metrics': ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision'], # default
        'topk':K,
        'valid_metric':VALID_METRIC          
    }


    evaluate_on_data_sections_bpr(model_name=model_name,
                                model_checkpoint_file=model_checkpoint_file,
                                current_dataset_name=current_dataset_name,
                                test_data_sections=test_data_sections,
                                parameter_dict=parameter_dict,
                                filename_version=FILENAME_VERSION)

### recall heatmap

In [ ]:
model_name = 'BPR'
results_matrix = get_results_matrix(model_name=model_name,
                                    models_versions=MODEL_VERSIONS,
                                    base_dataset_name=BASE_DATASET_NAME,
                                    save_path=SAVE_PATH,
                                    metric='recall@'+str(K[2]),
                                    filename_version=FILENAME_VERSION,
                                    part_shift_incl=False,
                                    test_full_data_sec=False)

recall_heatmap(results_matrix, round_point=4, title='test', filepath=None)

## Pop

In [ ]:
model_name = 'Pop'


for part in MODEL_VERSIONS:
    print('\n\n'+part)
    dataset_name=BASE_DATASET_NAME+part
    parameter_dict = {
        'dataset': dataset_name+'.inter',
        'use_gpu':USE_GPU,

        ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
        'seed':SEED,
        'state':'ERROR',
        'data_path': SAVE_PATH,
        # save_dataset (bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
        'checkpoint_dir':SAVE_PATH+dataset_name,
        # 'show_progress': SHOW_PROGRESS,
        'shuffle': SHUFFLE,

        ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
        'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
        # 'user_inter_num_interval':'[1,inf)',
        
        ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
        
        ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
        'eval_args': EVAL_ARGS,
        # 'metrics': ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision'], # default
        'topk':K,
        'valid_metric':VALID_METRIC          
    }

    recbole_train_evaluate_pop(model_name, dataset_name, parameter_dict, FILENAME_VERSION)


### evaluate on test set - pt1 without CS and on sectioned parts

In [ ]:
model_name = 'Pop'
model_checkpoint_versions = {'_pt1':model_name+'-Mar--2025_',
                             '_pt2':model_name+'-Mar--2025_',
                             '_pt3':model_name+'-Mar--2025_',
                             '_pt4':model_name+'-Mar--2025_'}


for part in MODEL_VERSIONS:
    print('\n\n'+part)

    current_dataset_name = BASE_DATASET_NAME+part

    # Checkpoint - 
    model_checkpoint_ver = model_checkpoint_versions[part]
    model_checkpoint_dir = SAVE_PATH+BASE_DATASET_NAME+part
    model_checkpoint_file = model_checkpoint_dir+'/'+model_checkpoint_ver+'.pth'


    test_data_sections = get_test_data_sections_with_names(model_version=part,
                                            base_dataset_name=BASE_DATASET_NAME,
                                            models_versions=MODEL_VERSIONS,
                                            pt1NoCS='NoCS')
    print(test_data_sections)
    

    parameter_dict = {
        'dataset': current_dataset_name+'.inter',
        'use_gpu':USE_GPU,

        ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
        'seed':SEED,
        'state':'ERROR',
        'data_path': SAVE_PATH,
        # save_dataset (bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
        'checkpoint_dir':model_checkpoint_dir,
        # 'show_progress': SHOW_PROGRESS,
        'shuffle': SHUFFLE,

        ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
        'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
        # 'user_inter_num_interval':'[1,inf)',
        
        ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
        
        ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
        'eval_args': EVAL_ARGS,
        # 'metrics': ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision'], # default
        'topk':K,
        'valid_metric':VALID_METRIC          
    }


    evaluate_on_data_sections_pop(model_name=model_name,
                                model_checkpoint_file=model_checkpoint_file,
                                current_dataset_name=current_dataset_name,
                                test_data_sections=test_data_sections,
                                parameter_dict=parameter_dict,
                                filename_version=FILENAME_VERSION)

### recall heatmap

In [ ]:
model_name = 'Pop'
results_matrix = get_results_matrix(model_name=model_name,
                                    models_versions=MODEL_VERSIONS,
                                    base_dataset_name=BASE_DATASET_NAME,
                                    save_path=SAVE_PATH,
                                    metric='recall@'+str(K[2]),
                                    filename_version=FILENAME_VERSION,
                                    part_shift_incl=False,
                                    test_full_data_sec=False)

recall_heatmap(results_matrix, round_point=4, title='test', filepath=None)

## NeuMF

In [ ]:
model_name = 'NeuMF'


for part in MODEL_VERSIONS:
    print('\n\n'+part)
    dataset_name=BASE_DATASET_NAME+part
    parameter_dict = {
        'dataset': dataset_name+'.inter',
        'use_gpu':USE_GPU,

        ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
        'seed':SEED,
        'state':'ERROR',
        'data_path': SAVE_PATH,
        # save_dataset (bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
        'checkpoint_dir':SAVE_PATH+dataset_name,
        # 'show_progress': SHOW_PROGRESS,
        'shuffle': SHUFFLE,

        ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
        'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
        # 'user_inter_num_interval':'[1,inf)',
        
        ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
        
        ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
        'eval_args': EVAL_ARGS,
        # 'metrics': ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision'], # default
        'topk':K,
        'valid_metric':VALID_METRIC          
    }

    recbole_train_evaluate_neumf(model_name, dataset_name, parameter_dict, FILENAME_VERSION)


### evaluate on test set - pt1 without CS and on sectioned parts

In [ ]:
model_name = 'NeuMF'
model_checkpoint_versions = {'_pt1':model_name+'-Mar--2025_',
                             '_pt2':model_name+'-Mar--2025_',
                             '_pt3':model_name+'-Mar--2025_',
                             '_pt4':model_name+'-Mar--2025_'}


for part in MODEL_VERSIONS:
    print('\n\n'+part)

    current_dataset_name = BASE_DATASET_NAME+part

    # Checkpoint - 
    model_checkpoint_ver = model_checkpoint_versions[part]
    model_checkpoint_dir = SAVE_PATH+BASE_DATASET_NAME+part
    model_checkpoint_file = model_checkpoint_dir+'/'+model_checkpoint_ver+'.pth'


    test_data_sections = get_test_data_sections_with_names(model_version=part,
                                            base_dataset_name=BASE_DATASET_NAME,
                                            models_versions=MODEL_VERSIONS,
                                            pt1NoCS='NoCS')
    print(test_data_sections)
    

    parameter_dict = {
        'dataset': current_dataset_name+'.inter',
        'use_gpu':USE_GPU,

        ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
        'seed':SEED,
        'state':'ERROR',
        'data_path': SAVE_PATH,
        # save_dataset (bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
        'checkpoint_dir':model_checkpoint_dir,
        # 'show_progress': SHOW_PROGRESS,
        'shuffle': SHUFFLE,

        ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
        'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
        # 'user_inter_num_interval':'[1,inf)',
        
        ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
        
        ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
        'eval_args': EVAL_ARGS,
        # 'metrics': ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision'], # default
        'topk':K,
        'valid_metric':VALID_METRIC          
    }


    evaluate_on_data_sections_neumf(model_name=model_name,
                                model_checkpoint_file=model_checkpoint_file,
                                current_dataset_name=current_dataset_name,
                                test_data_sections=test_data_sections,
                                parameter_dict=parameter_dict,
                                filename_version=FILENAME_VERSION)

### recall heatmap

In [ ]:
model_name = 'NeuMF'
results_matrix = get_results_matrix(model_name=model_name,
                                    models_versions=MODEL_VERSIONS,
                                    base_dataset_name=BASE_DATASET_NAME,
                                    save_path=SAVE_PATH,
                                    metric='recall@'+str(K[2]),
                                    filename_version=FILENAME_VERSION,
                                    part_shift_incl=False,
                                    test_full_data_sec=False)

recall_heatmap(results_matrix, round_point=4, title='test', filepath=None)

## ItemKNN

In [ ]:
model_name = 'ItemKNN'


for part in MODEL_VERSIONS:
    print('\n\n'+part)
    dataset_name=BASE_DATASET_NAME+part
    parameter_dict = {
        'dataset': dataset_name+'.inter',
        'use_gpu':USE_GPU,

        ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
        'seed':SEED,
        'state':'ERROR',
        'data_path': SAVE_PATH,
        # save_dataset (bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
        'checkpoint_dir':SAVE_PATH+dataset_name,
        # 'show_progress': SHOW_PROGRESS,
        'shuffle': SHUFFLE,

        ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
        'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
        # 'user_inter_num_interval':'[1,inf)',
        
        ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
        
        ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
        'eval_args': EVAL_ARGS,
        # 'metrics': ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision'], # default
        'topk':K,
        'valid_metric':VALID_METRIC          
    }

    recbole_train_evaluate_itemknn(model_name, dataset_name, parameter_dict, FILENAME_VERSION)


### evaluate on test set - pt1 without CS and on sectioned parts

In [ ]:
model_name = 'ItemKNN'
model_checkpoint_versions = {'_pt1':model_name+'-Mar--2025_',
                             '_pt2':model_name+'-Mar--2025_',
                             '_pt3':model_name+'-Mar--2025_',
                             '_pt4':model_name+'-Mar--2025_'}


for part in MODEL_VERSIONS:
    print('\n\n'+part)

    current_dataset_name = BASE_DATASET_NAME+part

    # Checkpoint - 
    model_checkpoint_ver = model_checkpoint_versions[part]
    model_checkpoint_dir = SAVE_PATH+BASE_DATASET_NAME+part
    model_checkpoint_file = model_checkpoint_dir+'/'+model_checkpoint_ver+'.pth'


    test_data_sections = get_test_data_sections_with_names(model_version=part,
                                            base_dataset_name=BASE_DATASET_NAME,
                                            models_versions=MODEL_VERSIONS,
                                            pt1NoCS='NoCS')
    print(test_data_sections)
    

    parameter_dict = {
        'dataset': current_dataset_name+'.inter',
        'use_gpu':USE_GPU,

        ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
        'seed':SEED,
        'state':'ERROR',
        'data_path': SAVE_PATH,
        # save_dataset (bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
        'checkpoint_dir':model_checkpoint_dir,
        # 'show_progress': SHOW_PROGRESS,
        'shuffle': SHUFFLE,

        ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
        'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
        # 'user_inter_num_interval':'[1,inf)',
        
        ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
        
        ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
        'eval_args': EVAL_ARGS,
        # 'metrics': ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision'], # default
        'topk':K,
        'valid_metric':VALID_METRIC          
    }


    evaluate_on_data_sections_itemknn(model_name=model_name,
                                model_checkpoint_file=model_checkpoint_file,
                                current_dataset_name=current_dataset_name,
                                test_data_sections=test_data_sections,
                                parameter_dict=parameter_dict,
                                filename_version=FILENAME_VERSION)

### recall heatmap

In [ ]:
model_name = 'ItemKNN'
results_matrix = get_results_matrix(model_name=model_name,
                                    models_versions=MODEL_VERSIONS,
                                    base_dataset_name=BASE_DATASET_NAME,
                                    save_path=SAVE_PATH,
                                    metric='recall@'+str(K[2]),
                                    filename_version=FILENAME_VERSION,
                                    part_shift_incl=False,
                                    test_full_data_sec=False)

recall_heatmap(results_matrix, round_point=4, title='test', filepath=None)